# CPG-RL **v3.6f**：智元 D1 Max · 平移抬腳頂點獎勵＋線性懲罰＋質心 DR · MJX · Colab GPU（2026-09-22）

在 v3.5f（`weights/cpg_rl_v3_5f_params.pkl`：九指令 0 摔、左轉 105% 原廠、航向 −32.7°→+4.5°）之上修四件事：
- **平移不踏步、靠側滑**：原廠週期模式下 `t_lift`／`t_mode`／`t_stance` 恆 0，policy 把每週期抬腳壓到 5–6 mm（零動作 14／11、原廠 21）→ 加 `t_step`：
  主動側兩腿**每週期抬腳頂點**對目標（21 mm，隨指令縮）的線性分數 × 1.5，只在平移族。
- **`t_drift`／`t_abadbias` 改死區＋線性**（`PEN_SHAPE="hinge"`）：二次式在 0.038 m/s 時每步只剩 0.058、推不到目標。
- **DR 加橫向質心偏移**（整機 ±5 mm 零均值）：模型質心偏左 1.5 mm 讓 v3.5f 右轉只有左轉的 79%，policy 學的是方向專屬補償。
- 驗收工具改預設無推力（之前每 2 s 一次的訓練推力污染了 roll／航向／vx 漂）。
設計：`docs/superpowers/specs/2026-09-22-cpg-rl-v3.6-step-apex-linear-penalties-design.md`；攤帳 `outputs/reward_audit_v36.md`。

**停損**：同 v3.5f —— 1 億步 `進度 yaw` < 3.5 或 len < 600 → 停；**另加**：1 億步 `step` < 0.3（平移族約佔回合四成、零動作名目 ≈ 0.6）→ 抬腳學不起來，停下來查 `W_STEP`。
`abad`／`drift`／`head` 三欄跨指令平均不可信（v3.5 spec §9.2），只看方向。
**注意**：G0 格的原地轉是開迴路原廠週期，**預期會倒**，那格只 assert 其他四個指令。


In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"

# 版本鎖死，不要放寬。brax 的預設 activation/分布由版本決定，而 activation
# 不匹配時 brax 載權重【不會報錯】，只會讓 policy 靜默錯亂。
# 這裡的版本 = 本機推論端 (task7/inference/) 的版本。
#
# jax<0.10：brax 0.14.2 的 ppo.train (train.py:756) 用 jax.device_put_replicated，
# 該 API 在 jax 0.10 已被移除（本機 jax 0.10.2 實測直接 AttributeError）。
# 用 jax[cuda12] 這個 extra 是為了讓 jaxlib 與 CUDA plugin 一起降到相容版本，
# 只寫 "jax<0.10" 會留下版本不合的 jaxlib/cuda plugin。
# ⚠️ 這一項無法在本機驗證：裝完務必看下一格印出的 devices 有沒有 cuda。
#    掉回 CPU 或裝不起來 → 回報，【不要】自行改 brax 版本（會動到 activation 預設值）。
!pip install -q "brax==0.14.2" "mujoco==3.10.0" "mujoco-mjx==3.10.0" "jax[cuda12]<0.10" mediapy
print("done")

In [ ]:
import jax
print("JAX", jax.__version__, "devices:", jax.devices())   # 要看到 cuda

# 版本斷言：不匹配當場停住，不要繞過。訓練跑完才發現行為對不上就白費了。
import brax, mujoco
print("brax", brax.__version__, "| mujoco", mujoco.__version__)
assert brax.__version__ == "0.14.2", (
    f"brax 版本為 {brax.__version__}，本機推論端是 0.14.2。"
    "版本不同會改變 make_ppo_networks 的預設 activation，"
    "而 activation 不匹配時 brax 載權重【不會報錯】，只會讓 policy 行為錯亂。"
    "請回到上一格重跑安裝（Colab 有時需要「執行階段 → 重新啟動工作階段」才會生效）。"
)
assert mujoco.__version__ == "3.10.0", (
    f"mujoco 版本為 {mujoco.__version__}，本機是 3.10.0。"
    "MJX 的接觸/求解器行為隨版本改變，訓練與推論不同版會讓步態對不上。"
)
# jax 0.10 移除了 device_put_replicated，而 brax 0.14.2 的 ppo.train 會用它。
# 在這裡早死，不要拖到訓練那一格編譯完才炸。
assert hasattr(jax, "device_put_replicated"), (
    f"jax {jax.__version__} 已移除 device_put_replicated，brax 0.14.2 的 ppo.train 會失敗。"
    "需要 jax<0.10。若 Colab 無法在此版本下取得 GPU 支援，請回報——"
    "換 brax 版本會改變 make_ppo_networks 的預設 activation，那會讓權重與本機推論端靜默不匹配。"
)
if not any(d.platform == "gpu" for d in jax.devices()):
    print("⚠️ 沒抓到 GPU。確認「執行階段 → 變更類型 → GPU」，"
          "以及上一格的 jax[cuda12]<0.10 是否把 CUDA 支援裝掉了。")
print("版本 OK")

In [ ]:
import os, subprocess, sys

REPO = "https://github.com/HGLLLLL/RBTDOG_SIM.git"
BRANCH = "main"
DEST = "rbtdog_sim"
if not os.path.exists(DEST):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, DEST], check=True)
sys.path.insert(0, f"{DEST}/task7/inference")
print("clone 到的 commit：",
      subprocess.run(["git", "-C", DEST, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)


In [ ]:
import jax, jax.numpy as jnp, numpy as np, mujoco
import rl_env_v3 as v3
env = v3.DualModeEnv(gains="factory", weights=v3.W36)
print("obs", env.obs_dim, "act", env.action_size, "gains", env.gains, "wheel_space", env.wheel_space)
assert env.wheel_pos is False
print("REF", {k: v for k, v in env.ref.items() if not hasattr(v, "shape")})
assert (env.obs_dim, env.action_size) == (88, 24)


In [ ]:
# ---- G0：零動作在五種指令下跑 10 s（同 diag/g0_v3.py --steps 500；門檻 spec §8.2／§9.1，本機結果 outputs/g0_v32_final.txt）
jit_reset, jit_step = jax.jit(env.reset), jax.jit(env.step)
def run(cmd, steps=500):
    s = jit_reset(jax.random.PRNGKey(0))
    s = s.replace(info={**s.info, "cmd": jnp.array(cmd), "cmd2": jnp.array(cmd), "t_switch": 10**6})
    a = jnp.zeros(v3.ACT_DIM); M = []
    for i in range(steps):
        s = jit_step(s, a); M.append({k: float(s.metrics[k]) for k in ("vx", "vy", "wz", "tau_pk", "roll", "clr_stance")} | {"done": float(s.done)})
        if float(s.done) > 0: break
    h = steps // 2; seg = M[h:] or M      # 5 s 前就倒（v3.3 TURN 開迴路預期）→ 用整段算，讓 fell 能回報而不是 max() 空序列炸掉
    return dict(vx=np.mean([m["vx"] for m in seg]), vy=np.mean([m["vy"] for m in seg]), yaw=np.degrees(np.mean([m["wz"] for m in seg])),
                tau=max(m["tau_pk"] for m in M), roll_std=float(np.std([m["roll"] for m in seg])), clr_stance=max(m["clr_stance"] for m in seg),
                fell=any(m["done"] > 0 for m in M))
R = {}
for name, cmd in (("WHEEL", (0.5, 0, 0)), ("ARC", (0.5, 0, 0.5)), ("TURN", (0, 0, 1.3)), ("LAT", (0, 0.08, 0)), ("DIAG", (0.3, 0.06, 0))):
    R[name] = r = run(cmd); print(name, {k: round(v, 3) if isinstance(v, float) else v for k, v in r.items()})
    assert name == "TURN" or not r["fell"], name          # v3.3：原地轉名目是開迴路原廠週期，預期會倒
assert R["WHEEL"]["vx"] >= 0.45 and R["WHEEL"]["clr_stance"] < 10
assert R["ARC"]["yaw"] >= 20
assert abs(R["LAT"]["vy"]) >= 0.05 and R["LAT"]["tau"] <= 85 and R["LAT"]["roll_std"] <= 1.5
assert R["DIAG"]["vx"] >= 0.15 and R["DIAG"]["vy"] >= 0.03
print("G0 全過 → 可以訓練")


In [ ]:
import functools, time
from brax.training.agents.ppo import train as ppo
from brax.training.agents.ppo import networks as ppo_networks

# 續訓：把 RESUME 指到已下載的 .pkl（存檔是 (normalizer, policy, value) 三元組，brax train.py:735 正好吃這個形狀）。
# None = 從頭訓。續訓時 TIMESTEPS 是「這一輪要再跑幾步」，不是總數。
RESUME = None
from brax.io import model as _bm
_restore = None if RESUME is None else _bm.load_params(RESUME)

env = v3.DualModeEnv(gains="factory", weights=v3.W36)
eval_env = v3.DualModeEnv(gains="factory", weights=v3.W36, ref=dict(cyc_amp_rand=False))      # 評估用固定幅度 0.7，曲線才可比
network_factory = functools.partial(ppo_networks.make_ppo_networks,
                                    policy_hidden_layer_sizes=(256, 256, 128), value_hidden_layer_sizes=(256, 256, 256))
TIMESTEPS = 200_000_000   # v3.3：2 億步（100M 約 60 分鐘）
train_fn = functools.partial(
    ppo.train, num_timesteps=TIMESTEPS, num_evals=20, episode_length=1000,
    num_envs=2048, batch_size=256, num_minibatches=32, unroll_length=20,
    num_updates_per_batch=4, learning_rate=3e-4, entropy_cost=1e-2,
    discounting=0.97, normalize_observations=True,
    learning_rate_schedule="ADAPTIVE_KL", desired_kl=0.01, learning_rate_schedule_min_lr=1e-5, learning_rate_schedule_max_lr=1e-3,
    network_factory=network_factory, randomization_fn=v3.make_domain_randomize("factory", com_y_mm=5.0), seed=0, eval_env=eval_env, restore_params=_restore)
_t0 = time.time(); rewards = []

def progress(step, metrics):
    r = float(metrics.get("eval/episode_reward", 0.0)); rewards.append((step, r))
    L = float(metrics.get("eval/avg_episode_length", 1.0)) or 1.0
    ps = lambda k: float(metrics.get(f"eval/episode_{k}", 0.0)) / L
    el = time.time() - _t0; rate = step / max(el, 1e-9)
    print(f"step {step:>11,} R {r:7.2f} | roll {ps('roll'):4.2f} pitch {ps('pitch'):4.2f} bias {ps('roll_bias'):+.2f} | "
          f"vxerr {ps('vxerr'):.3f} vyerr {ps('vyerr'):.3f} yawerr {ps('yawerr'):.3f} | 進度 yaw {ps('t_yawrel'):.2f}/8 vy {ps('t_vyrel'):.2f}/6 | s4 {ps('s4'):.2f} cyc {ps('cyc'):.2f} clr {ps('clr_step'):.0f}/{ps('clr_stance'):.0f} | "
          f"tau_pk {ps('tau_pk'):5.1f} err {ps('err_pk'):.3f} knee_v {ps('knee_v'):.1f} | abad {ps('abad_bias'):.1f}° drift {ps('vx_drift'):+.3f} head {ps('head_abs'):.1f}° | step {ps('t_step'):.2f} | len {L:.0f} | "
          f"{el:.0f}s → {TIMESTEPS / max(rate, 1) / 60:.0f} 分")

# 讀法：進度 yaw／vy ＝ 有該軸指令時沿指令方向的相對進度（分母 8／3），要往上；vxerr/vyerr/yawerr 往下、roll_bias 往 0、tau_pk < 58、len 往 1000。
make_inference_fn, params, _ = train_fn(environment=env, progress_fn=progress)
print("training done")


In [ ]:
import matplotlib.pyplot as plt
plt.plot([s for s, _ in rewards], [r for _, r in rewards], marker="o"); plt.xlabel("env steps"); plt.ylabel("eval reward"); plt.grid(True); plt.show()


In [ ]:
from brax.io import model
model.save_params("cpg_rl_v3_6f_params.pkl", params)
print("已存 cpg_rl_v3_6f_params.pkl → 下載放 task7/weights/；本機驗收 local_infer_v3.py（待寫）對標 outputs/ref_gait_dataset.md 的原廠數字")
